In [ ]:
!pip install -qU langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

print("Secret found:", api_key is not None)
print("Secret length:", len(api_key) if api_key else 0)

os.environ["OPENAI_API_KEY"] = api_key

Secret found: True
Secret length: 164


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

analysis_prompt = ChatPromptTemplate.from_template("""
You are a question analyzer for a Retrieval-Augmented Generation system.

Analyze the question and decide whether external knowledge retrieval
is needed.

Return exactly these fields:

use_rag: true or false
question_type: general, factual, multi-hop, definition, comparison
complexity: low, medium, high
reason: short explanation

Use RAG when the question is likely to require external knowledge.

Do not use RAG for:
- simple arithmetic
- casual conversation
- simple reasoning that does not require external information

Question:
{question}
""")

In [ ]:
def analyze_question(question):

    messages = analysis_prompt.invoke({
        "question": question
    })

    response = llm.invoke(messages)

    return response.content

In [ ]:
result = analyze_question(
    "When was Roskilde University founded?"
)

print(result)

use_rag: false  
question_type: factual  
complexity: low  
reason: The founding date of Roskilde University is a straightforward factual question that can be answered with basic knowledge.


In [ ]:
test_questions = [
    "What is 2 + 2?",
    "What is the capital of France?",
    "What is Roskilde University?",
    "Who founded Microsoft?",
    "What is the difference between Python and Java?",
    "Who was the mother of the person who founded Microsoft?",
    "Tell me a joke.",
    "What is the population of Denmark?",
    "Who is the primeminister of nepal?"
]

for question in test_questions:

    print("=" * 60)
    print("QUESTION:", question)

    result = analyze_question(question)

    print(result)

QUESTION: What is 2 + 2?
use_rag: false  
question_type: general  
complexity: low  
reason: The question involves simple arithmetic that does not require external knowledge.
QUESTION: What is the capital of France?
use_rag: false  
question_type: factual  
complexity: low  
reason: The question asks for a straightforward fact that is commonly known and does not require external knowledge retrieval.
QUESTION: What is Roskilde University?
use_rag: true  
question_type: definition  
complexity: low  
reason: The question asks for a definition or description of Roskilde University, which may require external knowledge to provide accurate and detailed information.
QUESTION: Who founded Microsoft?
use_rag: false  
question_type: factual  
complexity: low  
reason: The question asks for a straightforward fact that is widely known and does not require external knowledge retrieval.
QUESTION: What is the difference between Python and Java?
use_rag: true  
question_type: comparison  
complexity: